## Notation guide: Figure 1 and categorical evidence

For the selected sample $n$, `test_ev[k]` is evidence $e_{n,k}$, `test_alpha[k]` is $\alpha_{n,k}=e_{n,k}+1$, and `s`/`test_alpha.sum()` is $S_n=\sum_k\alpha_{n,k}$. The expected class probability is $\hat p_{n,k}=\alpha_{n,k}/S_n$, Eq. (4). The first four entries of `test_belief` are $b_{n,k}=e_{n,k}/S_n$, Eq. (5); its last entry is $u_n=K/S_n$, Eq. (6), with $K=4$.

`evi_ale` is $\mathbb{E}[\operatorname{Var}(y_{n,k}\mid p_{n,k})]$, Eqs. (20)–(21), and `evi_epi` is $\operatorname{Var}(p_{n,k})$, Eqs. (22)–(23), for the sleet event. `ens_ale` and `ens_epi` estimate the same decomposition using the sampled probabilities; printed square roots are standard deviations. The sleet-versus-rest marginal uses $(\alpha_{n,\mathrm{sleet}}, S_n-\alpha_{n,\mathrm{sleet}})$. The alternate illustration $(1-u_n)p_{n,\mathrm{sleet}}$ is a scaled random probability, not DST belief mass. The authors' external data and checkpoint are required to run these research figures.


In [100]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr 
import pandas as pd
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
from scipy.stats import dirichlet
from mlguess.keras.models import CategoricalDNN
import yaml
from bridgescaler import load_scaler
from keras.models import load_model
from matplotlib.patches import Polygon
%matplotlib inline

In [2]:
fn_config = "/glade/p/cisl/aiml/ai2es/winter_ptypes/models/evidential_unweighted/model.yml"
with open(fn_config) as cf:
    conf = yaml.load(cf, Loader=yaml.FullLoader)
input_features = (
    conf["TEMP_C"] + conf["T_DEWPOINT_C"] + conf["UGRD_m/s"] + conf["VGRD_m/s"]
)
output_features = conf["ptypes"]


In [56]:
from keras.layers import LeakyReLU
model = load_model("/glade/campaign/cisl/aiml/ai2es/winter_ptypes/models/evidential_unweighted/models/model_19.h5", custom_objects={"LeakyReLU":LeakyReLU})

/glade/work/dgagne/conda-envs/guess/lib/python3.10/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [57]:
test_data = pd.read_parquet("/glade/p/cisl/aiml/ai2es/winter_ptypes/models/evidential_unweighted/evaluate/test_19.parquet")

In [58]:
input_scaler_loc = os.path.join(conf["save_loc"], "scalers", "input.json")

input_scaler = load_scaler(input_scaler_loc)

In [59]:
scaled_input_x = input_scaler.transform(test_data[input_features])

In [60]:
test_preds = model.predict(scaled_input_x, batch_size=10000)

40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step


In [61]:
test_preds[:, 2].argmax()

317229

In [62]:
test_preds[:, 2].argsort()[-1:-10:-1]

array([317229, 317359, 317524, 317671, 317529, 317537, 317360, 317375,
       317856])

In [63]:
ps = np.arange(0, 1.01, 0.01)
all_ps = np.meshgrid(ps, 1-ps)
p_db = np.vstack([all_p.ravel() for all_p in all_ps]).T
p_simp = p_db[p_db.sum(axis=1)==1]

In [64]:
heights = np.arange(0, 5250.0, 250)
temps = np.linspace(0, -20, heights.size)
dewps = np.linspace(-10, -30, heights.size)
us = np.linspace(-10, 30, heights.size)
vs = np.linspace(-5, 10, heights.size)
px = np.arange(0, 1, 0.01)
pall = np.concatenate([px, 1-px]).reshape(2, px.size)
dp = dirichlet.pdf(pall, [10, 3])

In [ ]:
# Corrected marginal and moments; saved downstream figure outputs were cleared.
# Historical figures have not been rerun with the original data.
idx = 317529
temps = test_data.iloc[idx][conf["TEMP_C"]]
dewps = test_data.iloc[idx][conf["T_DEWPOINT_C"]]
us = test_data.iloc[idx][conf["UGRD_m/s"]]
vs = test_data.iloc[idx][conf["VGRD_m/s"]]
heights = np.arange(0, 5250.0, 250)
test_ev = np.maximum(test_preds[idx], 0)
test_alpha = test_ev + 1
# Sleet versus all other classes is the marginal of the full Dirichlet.
# Slicing alpha[2:] would condition on sleet/freezing rain instead.
sleet_class = 2
marginal_alpha = np.array([test_alpha[sleet_class],
                           test_alpha.sum() - test_alpha[sleet_class]])
dp = dirichlet.pdf(p_simp.T, marginal_alpha)
dp_ens = dirichlet.rvs(marginal_alpha, size=20)
test_belief = np.concatenate([test_ev / test_alpha.sum(), [test_ev.shape[0] / test_alpha.sum()]])

In [ ]:
s = test_alpha.sum()
evi_epi = (test_alpha[sleet_class] / s * (1 - test_alpha[sleet_class] / s)) / (s + 1)
evi_ale = test_alpha[sleet_class] / s - (test_alpha[sleet_class] / s) ** 2 - evi_epi
ens_epi = np.var(dp_ens[:, 0], ddof=0)
ens_ale = np.mean(dp_ens[:, 0] * (1 - dp_ens[:, 0]))
np.testing.assert_allclose(evi_ale + evi_epi,
                           (test_alpha[sleet_class] / s) * (1 - test_alpha[sleet_class] / s))
np.testing.assert_allclose(ens_ale + ens_epi, dp_ens[:, 0].mean() * (1 - dp_ens[:, 0].mean()))


In [ ]:
test_alpha

In [ ]:
total_evi_epi = 0
s = test_alpha.sum()
for i in range(4):
    total_evi_epi += (test_alpha[i] / s * (1 - test_alpha[i] / s)) / (s + 1)
print(np.sqrt(total_evi_epi))

In [ ]:
print(np.sqrt(evi_epi), np.sqrt(evi_ale))
print(np.sqrt(ens_epi), np.sqrt(ens_ale))

In [ ]:
evi_epi

In [ ]:
fig = plt.figure(tight_layout=True, figsize=(7, 6))
gs = gridspec.GridSpec(4, 2, height_ratios=[0.5, 0.2, 0.3, 0.1])
ax = fig.add_subplot(gs[0,0])
ax.plot(temps, heights / 1000, color='r', label="Temperature")
ax.plot(dewps, heights / 1000, color='g', label="Dewpoint", linestyle='--')
ax.set_xticks(np.arange(-20, 10, 5))
ax.set_xlim(-20, 5)
ax.legend(fontsize=8)
ax.set_xlabel(r"Temperature ($\degree$C)")
ax.set_ylabel("Height (km)")
ax.text(-22, 4.8, "(a)",
      fontsize=12, va='top', ha='right')
ax.plot(np.zeros(heights.shape[0]), heights / 1000, 'k--', lw=0.5)
ax.set_ylim(0, 5)

ax = fig.add_subplot(gs[0, 1])
cmap = plt.get_cmap("Set1", 5)
levels = ["0-1 km", "1-2 km", "2-3 km", "3-4 km", "4-5 km"]
hodo_ticks = np.arange(-10, 40, 10)
for i in range(5):
    ax.plot(us[4 * i: 4 * i +5], vs[4* i:4 * i + 5], marker='o', color=cmap.colors[i], label=levels[i])
ax.plot(np.zeros(hodo_ticks.shape), hodo_ticks, 'k--', lw=0.5)
ax.plot(hodo_ticks, np.zeros(hodo_ticks.shape),  'k--', lw=0.5)

ax.set_xlabel("Zonal Wind (m s$^{-1}$)")
ax.set_ylabel("Meridional Wind (m s$^{-1}$)")
ax.set_xticks(hodo_ticks)
ax.set_yticks(hodo_ticks)
ax.legend(fontsize=8)
ax.text(-25, 31, "(b)",
      fontsize=12, va='top', ha='right')

ax = fig.add_subplot(gs[1, :])
poly = Polygon([[-2.5,0],[2.5, 0], [5, 5], [-5, 5]], color="purple", clip_on=False)
ax.add_patch(poly)
ax.set_xlim(-5, 5)
ax.set_ylim(2, 5)
ax.text(0, 2.5, "Evidential or Ensemble Neural Networks", fontsize=12, color='w', ha="center", va="center")
ax.set_axis_off()
#s1 = Polygon([[-1, -1], [-0.5, -1], [-0.5, -0.5], [-1, -0.5]], clip_on=False)
#ax.add_patch(s1)
ax = fig.add_subplot(gs[2, :])
#ax.bar([0, 1], test_ev[2:][::-1], 0.02)
ax.plot(p_simp[:, 0],dp, color='tomato', label="Evidential PDF")
ax.hist(dp_ens[:, 0], np.arange(0, 1.05, .05), density=True, color='skyblue', label="Ensemble PDF")
ax.bar([test_alpha[sleet_class] / test_alpha.sum()], [dp.max()], 0.01, color='r', label="Evidential Expected Value")
ax.bar([dp_ens[:, 0].mean()], [dp.max()], 0.01, color='b', label="Ensemble Mean")
ax.legend(fontsize=8)
ax.set_xticks(np.arange(0, 1.1, 0.1))
ax.set_xlim(0, 1)
ax.set_xlabel("Probability of Sleet")
ax.set_ylabel("Density")
ax.text(-.04, 7, "(c)", fontsize=12, va='top', ha='center')

ax = fig.add_subplot(gs[3, :])
#ax.bar([0, 1], test_ev[2:][::-1], 0.02)
p_type_colors = ["green", "blue", "orange", "red", "purple"]
p_type_labels = ["Rain", "Snow", "Sleet", "Freezing\nRain", "Unknown"]
for b in range(test_belief.shape[0]):
    if b == 0:
        ax.barh(0.5, test_belief[b], height=1, left=0, color=p_type_colors[b])
    else:
        ax.barh(0.5, test_belief[b], height=1, left=test_belief.cumsum()[b-1], color=p_type_colors[b])
    if test_belief[b] > 0:
        if b == 0:
            ax.text((0 + test_belief.cumsum()[b]) / 2, 0.5, p_type_labels[b], 
                    ha="center", va="center", fontsize=8, color='white')
        else:
            ax.text((test_belief.cumsum()[b-1] + test_belief.cumsum()[b]) / 2, 0.5, p_type_labels[b], 
                    ha="center", va="center", fontsize=8, color='white')
    ax.set_ylim(0, 1)
    ax.set_xlim(0, 1)
    ax.set_yticks([])
    ax.set_xticks(np.arange(0, 1.1, 0.1))
ax.set_xlabel("Precipitation Type Belief Mass")
ax.text(-0.04, 0.8, "(d)", fontsize=12, va='top', ha='center')

plt.savefig("evidential_figure_1.png", dpi=300, bbox_inches="tight")
plt.savefig("evidential_figure_1.pdf", bbox_inches="tight")

In [ ]:
idx = 317529
temps = test_data.iloc[idx][conf["TEMP_C"]]
dewps = test_data.iloc[idx][conf["T_DEWPOINT_C"]]
us = test_data.iloc[idx][conf["UGRD_m/s"]]
vs = test_data.iloc[idx][conf["VGRD_m/s"]]
heights = np.arange(0, 5250.0, 250)
test_ev = np.maximum(test_preds[idx], 0)
test_alpha = test_ev + 1
dst_u = test_ev.shape[0] / test_alpha.sum()
# Sleet versus all other classes is the marginal of the full Dirichlet.
# Slicing alpha[2:] would condition on sleet/freezing rain instead.
sleet_class = 2
marginal_alpha = np.array([test_alpha[sleet_class],
                           test_alpha.sum() - test_alpha[sleet_class]])
dp = dirichlet.pdf(p_simp.T, marginal_alpha)
# Retain the alternate scaled-probability illustration, distinct from DST belief.
belief_scale = 1 - dst_u
dp_ens = dirichlet.rvs(marginal_alpha, size=20) * belief_scale
test_belief = np.concatenate([test_ev / test_alpha.sum(), [dst_u]])

In [ ]:
fig = plt.figure(tight_layout=True)
gs = gridspec.GridSpec(3, 2, height_ratios=[0.6, 0.3, 0.1])
ax = fig.add_subplot(gs[0,0])
ax.plot(temps, heights / 1000, color='r', label="Temperature")
ax.plot(dewps, heights / 1000, color='g', label="Dewpoint", linestyle='--')
ax.set_xticks(np.arange(-20, 10, 5))
ax.set_xlim(-20, 5)
ax.legend(fontsize=8)
ax.set_xlabel("Temperature (C)")
ax.set_ylabel("Height (km)")
ax.plot(np.zeros(heights.shape[0]), heights / 1000, 'k--', lw=0.5)
ax.set_ylim(0, 5)
ax = fig.add_subplot(gs[0, 1])
cmap = plt.get_cmap("Set1", 5)
levels = ["0-1 km", "1-2 km", "2-3 km", "3-4 km", "4-5 km"]
hodo_ticks = np.arange(-10, 40, 10)
for i in range(5):
    ax.plot(us[4 * i: 4 * i +5], vs[4* i:4 * i + 5], marker='o', color=cmap.colors[i], label=levels[i])
ax.plot(np.zeros(hodo_ticks.shape), hodo_ticks, 'k--', lw=0.5)
ax.plot(hodo_ticks, np.zeros(hodo_ticks.shape),  'k--', lw=0.5)

ax.set_xlabel("West-East Wind (m/s)")
ax.set_ylabel("South-North Wind (m/s)")
ax.set_xticks(hodo_ticks)
ax.set_yticks(hodo_ticks)
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[1, :])
#ax.bar([0, 1], test_ev[2:][::-1], 0.02)
# For Z = c * P, the density is f_Z(z) = f_P(z/c) / c.
if belief_scale > 0:
    ax.plot(p_simp[:, 0] * belief_scale, dp / belief_scale,
            color='tomato', label="Scaled evidential PDF")
    density_peak = dp.max() / belief_scale
else:
    ax.axvline(0, color='tomato', label="Point mass at zero")
    density_peak = 1.0
ax.hist(dp_ens[:, 0], np.arange(0, 1.05, .05), density=True, color='skyblue', label="Ensemble PDF")
ax.bar([belief_scale * test_alpha[sleet_class] / test_alpha.sum()], [density_peak], 0.01, color='r', label="Evidential Expected Value")
ax.bar([dp_ens[:, 0].mean()], [density_peak], 0.01, color='b', label="Ensemble Mean")
ax.legend(fontsize=8)
ax.set_xticks(np.arange(0, 1.1, 0.1))
ax.set_xlim(0, 1)
ax.set_xlabel("Scaled sleet probability, (1-u) p (not DST belief mass)")
ax.set_ylabel("Density")

ax = fig.add_subplot(gs[2, :])
#ax.bar([0, 1], test_ev[2:][::-1], 0.02)
p_type_colors = ["green", "blue", "orange", "red", "purple"]
p_type_labels = ["Rain", "Snow", "Sleet", "Freezing\nRain", "Unknown"]
for b in range(test_belief.shape[0]):
    if b == 0:
        ax.barh(0.5, test_belief[b], height=1, left=0, color=p_type_colors[b])
    else:
        ax.barh(0.5, test_belief[b], height=1, left=test_belief.cumsum()[b-1], color=p_type_colors[b])
    if test_belief[b] > 0:
        if b == 0:
            ax.text((0 + test_belief.cumsum()[b]) / 2, 0.5, p_type_labels[b], 
                    ha="center", va="center", fontsize=8, color='white')
        else:
            ax.text((test_belief.cumsum()[b-1] + test_belief.cumsum()[b]) / 2, 0.5, p_type_labels[b], 
                    ha="center", va="center", fontsize=8, color='white')
    ax.set_ylim(0, 1)
    ax.set_xlim(0, 1)
    ax.set_yticks([])
    ax.set_xticks(np.arange(0, 1.1, 0.1))
ax.set_xlabel("Precipitation Type Belief Mass")
plt.savefig("evidential_figure_1_bel.png", dpi=300, bbox_inches="tight")
plt.savefig("evidential_figure_1_bel.pdf", bbox_inches="tight")